# 1. ライブラリインポート・定数などの設定

In [ ]:
import os
import sys
import json
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# データパスの設定
import os
if os.path.exists("data/ticker_dictionary.json"):
    DB_PATH = "data/stock_data.db"
    TICKER_DICT_PATH = "data/ticker_dictionary.json"
else:
    DB_PATH = "../data/stock_data.db"
    TICKER_DICT_PATH = "../data/ticker_dictionary.json"

# ノートブックでのグラフ表示用設定
%matplotlib inline
plt.rcParams['font.family'] = 'sans-serif' # 環境に合わせて変更してください

print("Libraries imported successfully.")


# 2. ターゲット設定（対象銘柄と期間）

In [ ]:
# 検出したい特定銘柄群（銘柄コードのリスト）
TARGET_CODES = ['6976', '6762', '6981'] # 太陽誘電, TDK, 村田製作所

# 特定しておきたかった期間
START_DATE = '2026-04-01'
END_DATE = '2026-05-25'

print(f"ターゲット銘柄: {TARGET_CODES}")
print(f"対象期間: {START_DATE} 〜 {END_DATE}")


# 3. データの読み込みと共通テーマ（タグ）の特定

In [ ]:
# タグマスターの読み込み
with open(TICKER_DICT_PATH, 'r', encoding='utf-8') as f:
    ticker_dict = json.load(f)

code_to_name = {}
code_to_tags = {}

for item in ticker_dict:
    code = item['code']
    code_to_name[code] = item['name']
    code_to_tags[code] = item.get('tags', [])

# ターゲット銘柄の存在確認と共通タグの抽出
target_names = []
common_tags = None

for code in TARGET_CODES:
    name = code_to_name.get(code, "Unknown")
    target_names.append(name)
    tags = set(code_to_tags.get(code, []))
    
    if common_tags is None:
        common_tags = tags
    else:
        common_tags = common_tags.intersection(tags)

print("【ターゲット銘柄】")
for code, name in zip(TARGET_CODES, target_names):
    print(f"- {name} ({code}) : {code_to_tags.get(code, [])}")

print("\n【ターゲット銘柄の共通タグ（積集合）】")
if common_tags:
    for tag in common_tags:
        print(f"★ {tag}")
else:
    print("共通タグは見つかりませんでした。")


# 4. 株価データの取得と動意指標の計算

In [ ]:
def fetch_and_prepare_target_data(db_path, target_codes):
    conn = sqlite3.connect(db_path)
    
    placeholders = ','.join('?' for _ in target_codes)
    query = f"""
    SELECT date, code, close, volume
    FROM daily_prices 
    WHERE code IN ({placeholders})
    ORDER BY code, date
    """
    
    df = pd.read_sql_query(query, conn, params=target_codes)
    conn.close()
    
    # 銘柄コードを文字列型に統一
    df['code'] = df['code'].astype(str)
    
    # 日付型への変換
    df['date_dt'] = pd.to_datetime(df['date'], format='%Y%m%d')
    
    # 数値列の型変換
    for col in ['close', 'volume']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # 5日・25日移動平均出来高の計算
    df['vol_5d_avg'] = df.groupby('code')['volume'].rolling(window=5, min_periods=1).mean().reset_index(0, drop=True)
    df['vol_25d_avg'] = df.groupby('code')['volume'].rolling(window=25, min_periods=1).mean().reset_index(0, drop=True)
    
    # 出来高急増倍率 (vol_ratio)
    df['vol_ratio'] = df['vol_5d_avg'] / df['vol_25d_avg']
    
    # 過去250日高値の計算
    df['high_250d'] = df.groupby('code')['close'].rolling(window=250, min_periods=1).max().reset_index(0, drop=True)
    
    # 高値圏位置 (price_vs_high)
    df['price_vs_high'] = df['close'] / df['high_250d']
    
    return df

# 全データを取得して指標を計算
df_all = fetch_and_prepare_target_data(DB_PATH, TARGET_CODES)

# 対象期間でフィルタリング
start_dt = pd.to_datetime(START_DATE)
end_dt = pd.to_datetime(END_DATE)

df_target = df_all[(df_all['date_dt'] >= start_dt) & (df_all['date_dt'] <= end_dt)].copy()

print(f"データ取得完了: {len(df_target)} 件のレコードを取得しました。")


# 5. 株価と出来高モメンタムの推移可視化

In [ ]:
# 銘柄ごとにプロット
for code in TARGET_CODES:
    df_plot = df_target[df_target['code'] == code].copy()
    if df_plot.empty:
        print(f"{code} のデータがありません。")
        continue
        
    name = code_to_name.get(code, "Unknown")
    
    fig, ax1 = plt.subplots(figsize=(12, 5))
    
    # 株価のプロット（左軸）
    color = 'tab:blue'
    ax1.set_xlabel('Date')
    ax1.set_ylabel('Close Price', color=color)
    ax1.plot(df_plot['date_dt'], df_plot['close'], color=color, label='Close')
    ax1.tick_params(axis='y', labelcolor=color)
    
    # 出来高倍率のプロット（右軸）
    ax2 = ax1.twinx()
    color = 'tab:red'
    ax2.set_ylabel('Volume Ratio (5d/25d)', color=color)
    ax2.plot(df_plot['date_dt'], df_plot['vol_ratio'], color=color, linestyle='--', label='Vol Ratio')
    ax2.tick_params(axis='y', labelcolor=color)
    
    # 基準線（出来高倍率2.0）
    ax2.axhline(y=2.0, color='gray', linestyle=':', alpha=0.7)
    
    plt.title(f"{name} ({code}) - Price & Volume Momentum")
    fig.tight_layout()
    plt.show()


# 6. 検知に必要なパラメータ（閾値）のボトムアップ逆算

In [ ]:
print(f"【{START_DATE} 〜 {END_DATE} の期間において、各銘柄を検知するために必要だった最低条件】\n")

min_required_vol_ratio = float('inf')
min_required_price_high = float('inf')
max_possible_vol_ratio = 0
max_possible_price_high = 0

for code in TARGET_CODES:
    df_code = df_target[df_target['code'] == code]
    if df_code.empty:
        continue
    
    name = code_to_name.get(code, "Unknown")
    
    # 期間中の最大出来高倍率
    max_vol_ratio = df_code['vol_ratio'].max()
    
    # 期間中の最高値圏位置
    max_price_vs_high = df_code['price_vs_high'].max()
    
    print(f"◆ {name} ({code})")
    print(f"   - 期間中最大 出来高急増倍率 (vol_ratio) : {max_vol_ratio:.2f} 倍")
    print(f"   - 期間中最大 高値圏位置 (price_vs_high): {max_price_vs_high:.3f} ({(max_price_vs_high*100):.1f}%)")
    print(f"   => 少なくとも VOL_RATIO={max_vol_ratio:.1f} 以下, PRICE_HIGH={max_price_vs_high:.2f} 以下の設定なら期間内に1度は網にかかります。\n")
    
    # ターゲット群全体を「全員」少なくとも1回は検知するための共通閾値（最も厳しい条件）
    min_required_vol_ratio = min(min_required_vol_ratio, max_vol_ratio)
    min_required_price_high = min(min_required_price_high, max_price_vs_high)
    
    # ターゲット群の誰か1人でも検知できればよい場合の閾値（最も緩い条件）
    max_possible_vol_ratio = max(max_possible_vol_ratio, max_vol_ratio)
    max_possible_price_high = max(max_possible_price_high, max_price_vs_high)


print("=" * 60)
print("🎯 【結論】指定期間にターゲット銘柄群を検知するための推奨パラメータ")
print("=" * 60)
print(f"【全員を少なくとも1日は検知したい場合（保守的設定）】")
print(f" => VOL_RATIO を {min_required_vol_ratio:.2f} 以下 に設定する")
print(f" => PRICE_HIGH を {min_required_price_high:.3f} 以下 に設定する")
print()
print(f"【誰か1銘柄でも引っかかればよい場合（緩い設定）】")
print(f" => VOL_RATIO を {max_possible_vol_ratio:.2f} 以下 に設定する")
print(f" => PRICE_HIGH を {max_possible_price_high:.3f} 以下 に設定する")
print("=" * 60)
